In [5]:
# Instalação das dependências no Google Colab
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!pip install webdriver_manager


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Ign:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy Release
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading pack

In [11]:

import csv
import os
import time
from datetime import date, datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException, NoSuchElementException, ElementClickInterceptedException
import shutil
from google.colab import drive

# Montagem do Google Drive para salvar os dados
drive.mount('/content/drive')

# Definir o caminho para salvar o arquivo CSV no Google Drive
output_dir = '/content/drive/MyDrive/scraping_outputs'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
output_path = os.path.join(output_dir, 'tumelero_produtos.csv')

# Configuração do WebDriver para uso no Colab
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

# Inicializa o WebDriver
driver = webdriver.Chrome(options=chrome_options)

# Abre a página do Tumelero
driver.get('https://www.tumelero.com.br/pisos-e-revestimentos/pisos/pisos-ceramicos')

# Espera até que os elementos de produtos estejam visíveis
try:
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'li[data-smarthintfeature="search"]'))
    )
except TimeoutException:
    print("Tempo de espera expirou. Não foi possível encontrar os produtos.")
    driver.quit()
    exit()

# Função para extrair dados da página
def extrair_dados():
    dados = []
    pagina_atual = 1

    while True:
        try:
            produtos = driver.find_elements(By.CSS_SELECTOR, 'li[data-smarthintfeature="search"]')
            print(f"Total de produtos encontrados nesta página: {len(produtos)}")

            for produto in produtos:
                try:
                    nome = produto.find_element(By.CSS_SELECTOR, 'h2.x-shelf__title').text
                    preco = produto.find_element(By.CSS_SELECTOR, 'span.x-shelf__best-price strong').text
                    imagem = produto.find_element(By.CSS_SELECTOR, 'div.x-shelf__img-container img').get_attribute('src')
                    link = produto.find_element(By.CSS_SELECTOR, 'div.x-shelf__img-container a').get_attribute('href')

                    dados.append({
                        'nome': nome,
                        'preco': preco,
                        'imagem': imagem,
                        'link': link
                    })
                except NoSuchElementException as e:
                    print(f"Elemento não encontrado dentro do produto: {e}")

            try:
                # Espera até que o botão "Mais produtos" esteja clicável e sem overlay
                WebDriverWait(driver, 20).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, 'div.father-loadmore-smarthint button.cf-load-more'))
                )

                botao_mais = driver.find_element(By.CSS_SELECTOR, 'div.father-loadmore-smarthint button.cf-load-more')
                driver.execute_script("arguments[0].scrollIntoView(true);", botao_mais)  # Scroll para o botão

                # Tenta clicar no botão e lida com overlays
                try:
                    botao_mais.click()
                    time.sleep(2)  # Espera o carregamento dos produtos
                except ElementClickInterceptedException:
                    print("Overlay ou outro elemento interceptou o clique. Tentando novamente...")
                    time.sleep(1)
                    driver.execute_script("arguments[0].click();", botao_mais)

                # Verifica se a página foi atualizada
                pagina_atual += 1
                print(f"Next page: {pagina_atual}")

            except NoSuchElementException:
                print("Não foi possível encontrar o botão 'Mais produtos'.")
                break
            except StaleElementReferenceException:
                print("O botão 'Mais produtos' não está mais disponível. Encerrando coleta.")
                break

        except NoSuchElementException as e:
            print(f"Erro ao encontrar produtos: {e}")
            break

    return dados

# Execução principal
dados = extrair_dados()

# Verifica se dados foram coletados
if not dados:
    print("Nenhum dado coletado.")
else:
    # Exibe os dados coletados no console
    for dado in dados:
        print(f"Nome: {dado['nome']}, Preço: {dado['preco']}, Imagem: {dado['imagem']}, Link: {dado['link']}")

# Fechando o navegador
driver.quit()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total de produtos encontrados nesta página: 36
Overlay ou outro elemento interceptou o clique. Tentando novamente...
Next page: 2
Total de produtos encontrados nesta página: 36
Overlay ou outro elemento interceptou o clique. Tentando novamente...
Next page: 3
Total de produtos encontrados nesta página: 72
Overlay ou outro elemento interceptou o clique. Tentando novamente...
Next page: 4
Total de produtos encontrados nesta página: 108
Overlay ou outro elemento interceptou o clique. Tentando novamente...
Next page: 5
Total de produtos encontrados nesta página: 144
Elemento não encontrado dentro do produto: Message: no such element: Unable to locate element: {"method":"css selector","selector":"span.x-shelf__best-price strong"}
  (Session info: chrome-headless-shell=127.0.6533.119); For documentation on this error, please visit: https://www.selenium.dev/document

KeyboardInterrupt: 